# RL-LLM Tutor — Kaggle 2xT4 GPU Version (Full Pipeline)
> Generates all data from scratch — no dataset import needed.  
> Uses 2xT4 GPUs, batch=512, 15 epochs.  
> Runs: Install → Setup → Write Modules → API Key → Generate D → Aug D+ → Aug D++ → Train → Eval → Plots


---
## Cell 1 - Install Dependencies
Run once. Restart kernel after, then run from Cell 2.

In [1]:
# Cell 1: Install dependencies (Kaggle)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "gym", "gymnasium", "dopamine-rl"], capture_output=True)

result = subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "d3rlpy==2.4.0",
    "gymnasium==0.29.1",
    "sentence-transformers==2.7.0",
    "datasets==2.20.0",
    "scikit-learn>=1.3",
    "scipy>=1.11",
    "numpy>=1.24",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "seaborn>=0.12",
    "tqdm",
    "requests",
], capture_output=True, text=True)

if result.returncode != 0:
    print("INSTALL FAILED:\n", result.stderr[-3000:])
else:
    print("All packages installed.")
    print("If first run: Kernel -> Restart & Run All (skip Cell 1 after restart).")


All packages installed.
If first run: Kernel -> Restart & Run All (skip Cell 1 after restart).


---
## Cell 2 - Project Directories
Creates all folders. No Kaggle dataset import needed.

In [2]:
# Cell 2: Directory setup (generate everything from scratch)
import os, sys

BASE       = "/kaggle/working/rl_llm_tutor"
SRC_DIR    = f"{BASE}/src"
DATA_DIR   = f"{BASE}/data"
MODELS_DIR = f"{BASE}/models"
OUTPUT_DIR = f"{BASE}/output"

for d in [SRC_DIR, DATA_DIR, MODELS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Folders created:")
for d in [BASE, SRC_DIR, DATA_DIR, MODELS_DIR, OUTPUT_DIR]:
    print(f"  {d}")


Folders created:
  /kaggle/working/rl_llm_tutor
  /kaggle/working/rl_llm_tutor/src
  /kaggle/working/rl_llm_tutor/data
  /kaggle/working/rl_llm_tutor/models
  /kaggle/working/rl_llm_tutor/output


---
## Cell 3 - GPU Verification (2xT4)

In [3]:
# Cell 3: Verify 2xT4 GPUs
import torch, functools, torch.serialization as _ts

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  ({props.total_memory/1e9:.1f} GB)")

if torch.cuda.device_count() >= 2:
    print("\n[OK] 2xT4 detected.")
elif torch.cuda.device_count() == 1:
    print("\n[WARN] Only 1 GPU. Set Kaggle accelerator to GPU T4 x2.")
else:
    print("\n[ERROR] No GPU. Set Kaggle accelerator to GPU T4 x2 in Settings.")

_real_torch_load = _ts.load.__wrapped__ if hasattr(_ts.load, '__wrapped__') else _ts.load

@functools.wraps(_real_torch_load)
def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    if pickle_module is not None:
        return _real_torch_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)
    return _real_torch_load(f, map_location=map_location, **kwargs)

_ts.load = _patched_load
torch.load = _patched_load
print("[OK] torch.load patched for weights_only=False (d3rlpy compatibility)")


PyTorch  : 2.10.0+cu128
CUDA     : 12.8
GPU count: 2
  GPU 0: Tesla T4  (15.6 GB)
  GPU 1: Tesla T4  (15.6 GB)

[OK] 2xT4 detected.
[OK] torch.load patched for weights_only=False (d3rlpy compatibility)


---
## Cell 4 - Write src/config.py
All hyperparameters. 15 epochs, batch=512.

In [4]:
# Cell 4: Write src/config.py
import os
os.makedirs(SRC_DIR, exist_ok=True)

config_src = '''import os, random
import numpy as np

BASE        = "/kaggle/working/rl_llm_tutor"
DATA_DIR    = f"{BASE}/data"
MODELS_DIR  = f"{BASE}/models"
OUTPUT_DIR  = f"{BASE}/output"
SRC_DIR     = f"{BASE}/src"
for _d in [DATA_DIR, MODELS_DIR, OUTPUT_DIR, SRC_DIR]:
    os.makedirs(_d, exist_ok=True)

MISTRAL_MODEL      = "mistral-small-latest"
MAX_TURNS          = 8
N_DIALOGUES        = 2000
N_AUG_TARGET       = 3000
N_AUG_SAMPLES      = N_AUG_TARGET - N_DIALOGUES
N_AUG2_SAMPLES     = 500
N_EVAL             = 15
DISCOUNT           = 0.9
STATE_DIM          = 25
N_ACTIONS          = 4
ACTION_NAMES       = ["Instruct", "Encourage", "Refocus", "Question"]
SEED               = 42
N_WORKERS          = 4
CKPT_EVERY         = 200

STEPS_PER_EPOCH    = 10_000
N_TRAIN_STEPS      = 15 * STEPS_PER_EPOCH
SUPER_N_STEPS      = 15 * STEPS_PER_EPOCH

BATCH_SIZE         = 512
CQL_ALPHA          = 4.0
CQL_LR             = 5e-5
BC_LR              = 1e-3
DQN_LR             = 5e-5
SUPER_CQL_ALPHA    = 8.0
SUPER_CQL_LR       = 3e-5
SUPER_SEEDS        = [42, 7, 13]
REWARD_SHAPE_BONUS = 0.5

random.seed(SEED)
np.random.seed(SEED)
'''

with open(f"{SRC_DIR}/config.py", "w") as f:
    f.write(config_src)

from config import *
print("config.py written (15 epochs, batch=512)")
print(f"  DISCOUNT={DISCOUNT}, BATCH_SIZE={BATCH_SIZE}, N_EVAL={N_EVAL}")
print(f"  SUPER_SEEDS={SUPER_SEEDS}")


config.py written (15 epochs, batch=512)
  DISCOUNT=0.9, BATCH_SIZE=512, N_EVAL=15
  SUPER_SEEDS=[42, 7, 13]


---
## Cell 5 - Write src/llm_utils.py

In [5]:
# Cell 5: Write src/llm_utils.py
import os
os.makedirs(SRC_DIR, exist_ok=True)

llm_src = r'''
import os, time, random, requests, re
import concurrent.futures

def call_llm(prompt, max_tokens=80, temperature=0.7, retries=6, system=None):
    api_key = os.environ.get("MISTRAL_API_KEY", "")
    if not api_key:
        return _mock_response(prompt)
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    payload = {
        "model": os.environ.get("MISTRAL_MODEL", "mistral-small-latest"),
        "messages": msgs, "max_tokens": max_tokens, "temperature": temperature
    }
    for attempt in range(retries):
        try:
            r = requests.post("https://api.mistral.ai/v1/chat/completions",
                              headers=headers, json=payload, timeout=25)
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            elif r.status_code == 429:
                time.sleep(min(64, (2 ** attempt) * 1.5 + random.uniform(0.3, 2.5)))
            elif r.status_code >= 500:
                time.sleep(2 ** attempt)
            else:
                return ""
        except requests.exceptions.Timeout:
            time.sleep(2 ** attempt + random.uniform(0, 1))
        except Exception:
            if attempt == retries - 1:
                return ""
            time.sleep(2 ** attempt)
    return ""

def call_mistral(prompt, model="mistral-small-latest", max_tokens=256, temperature=0.7, retries=3):
    return call_llm(prompt, max_tokens=max_tokens, temperature=temperature, retries=retries)

def _mock_response(prompt):
    import random
    templates = [
        "Let me help you work through this step by step.",
        "That is a great question! Let us think about it carefully.",
        "You are on the right track. Consider breaking the problem into smaller parts.",
        "Try thinking about what information you already know.",
    ]
    return random.choice(templates)

def extract_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    return nums[-1] if nums else None

def call_llm_batch(prompts, max_tokens=80, temperature=0.7, n_workers=8, system=None):
    results = [""] * len(prompts)
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as exe:
        fut_map = {exe.submit(call_llm, p, max_tokens, temperature, 6, system): i
                   for i, p in enumerate(prompts)}
        for fut in concurrent.futures.as_completed(fut_map):
            idx = fut_map[fut]
            try:
                results[idx] = fut.result()
            except Exception:
                results[idx] = ""
    return results

def check_api_key():
    resp = call_llm("Reply OK", max_tokens=5)
    ok = bool(resp and len(resp) > 0)
    print(f"  API key: {chr(39)+chr(39)+chr(39)}{'PASSED' if ok else 'FAILED'}{chr(39)+chr(39)+chr(39)} | response={resp!r}")
    return ok
'''

with open(f"{SRC_DIR}/llm_utils.py", "w") as f:
    f.write(llm_src.lstrip())
print("llm_utils.py written (call_mistral alias + extract_number + mock fallback)")


llm_utils.py written (call_mistral alias + extract_number + mock fallback)


---
## Cell 6 - Write src/environment.py
Gymnasium TutorEnv with real student/tutor simulation via Mistral.

In [6]:
# Cell 6: Write src/environment.py
import os
os.makedirs(SRC_DIR, exist_ok=True)

env_src = r'''
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from config import STATE_DIM, N_ACTIONS, DISCOUNT, MAX_TURNS
from llm_utils import call_mistral, extract_number

class TutorEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, problems, max_turns=MAX_TURNS):
        super().__init__()
        self.problems   = problems
        self.max_turns  = max_turns
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf,
                                            shape=(STATE_DIM,), dtype=np.float32)
        self.action_space      = spaces.Discrete(N_ACTIONS)
        self._problem  = None
        self._state    = None
        self._turn     = 0
        self._history  = []
        self._dialogue = ""

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._problem  = self.np_random.choice(self.problems) if hasattr(self, "np_random") \
                         else self.problems[np.random.randint(len(self.problems))]
        self._turn     = 0
        self._history  = []
        self._dialogue = f"Student: I need help with: {self._problem['question']}"
        self._state    = self._encode_state(False)
        return self._state.copy(), {}

    def step(self, action):
        tutor_resp   = self._tutor_response(action)
        self._dialogue += f"\nTutor: {tutor_resp}"
        student_resp = self._student_response()
        self._dialogue += f"\nStudent: {student_resp}"
        self._history.append({"action": int(action), "tutor": tutor_resp, "student": student_resp})
        self._turn += 1
        solved = self._check_solved(student_resp)
        if solved:
            reward = 1.0
        elif self._turn >= self.max_turns:
            reward = -1.0
        else:
            reward = 0.0
        terminated = solved or self._turn >= self.max_turns
        self._state = self._encode_state(solved)
        return self._state.copy(), reward, terminated, False, {"solved": solved}

    def _tutor_response(self, action):
        styles = [
            "Give ONE short hint about what concept to think about, without showing any calculations or steps",
            "Encourage the student warmly without giving any hints about the solution",
            "Gently redirect the student back to focus on the problem without giving hints",
            "Ask the student ONE short question to check their understanding, without revealing anything"
        ]
        hist = "\n".join([f"Tutor: {h['tutor']}\nStudent: {h['student']}"
                           for h in self._history[-2:]])
        q = self._problem["question"]
        prompt = (f"You are a math tutor helping a struggling 6th grade student.\n"
                  f"Your goal: {styles[action]}.\n"
                  f"Problem: {q}\n"
                  f"Recent history: {hist if hist else 'None'}\n"
                  f"IMPORTANT: Do NOT show any calculations, numbers, or solution steps. "
                  f"Do NOT reveal the answer or intermediate values. "
                  f"Keep response under 40 words.")
        resp = call_mistral(prompt, max_tokens=60, temperature=0.7)
        return resp if resp else "Let me help you think through this."

    def _student_response(self):
        q = self._problem["question"]
        prompt = (f"You are a 6th grade student who is NOT good at math and struggles with word problems.\n"
                  f"You often make arithmetic mistakes and get confused easily.\n"
                  f"You only give a final numerical answer if you are very confident after multiple hints.\n"
                  f"Problem: {q}\n"
                  f"Conversation so far:\n{self._dialogue[-1500:]}\n"
                  f"Respond in 1-2 sentences like a confused student. "
                  f"Only state a number if you are fully confident. Otherwise express confusion or ask for help.")
        resp = call_mistral(prompt, max_tokens=60, temperature=0.9)
        return resp if resp else "I am not sure how to do this."

    def _check_solved(self, student_resp):
        ans = extract_number(self._problem.get("answer", ""))
        got = extract_number(student_resp)
        if ans is None or got is None:
            return False
        try:
            return abs(float(ans) - float(got)) < 0.5
        except ValueError:
            return False

    def _encode_state(self, solved):
        s = np.zeros(STATE_DIM, dtype=np.float32)
        s[0] = self._turn / max(self.max_turns, 1)
        s[1] = float(solved)
        s[2] = len(self._history) / max(self.max_turns, 1)
        if self._history:
            for i, h in enumerate(self._history[-4:]):
                s[3 + i] = h["action"] / 3.0
        tutor_q   = sum(1 for h in self._history if "?" in h.get("tutor", ""))
        student_q = sum(1 for h in self._history if "?" in h.get("student", ""))
        s[7] = min(tutor_q / 10.0, 1.0)
        s[8] = min(student_q / 10.0, 1.0)
        return s
'''

with open(f"{SRC_DIR}/environment.py", "w") as f:
    f.write(env_src.lstrip())
print("environment.py written (TutorEnv with real student/tutor simulation, fixed reward: +1/-1/0)")


environment.py written (TutorEnv with real student/tutor simulation, fixed reward: +1/-1/0)


---
## Cell 7 - Write src/state_extractor.py
25-dim state: 18 MiniLM dims + 7 hand-crafted features. Thread-safe, GPU-aware.

In [7]:
# Cell 7: Write src/state_extractor.py (thread-safe, meta-tensor safe)
import os, textwrap
os.makedirs(SRC_DIR, exist_ok=True)

src = textwrap.dedent('''
    import re, threading
    import numpy as np

    STATE_DIM = 25
    _encoder  = None
    _enc_lock = threading.Lock()

    _OFFTOPIC = ["anyway", "by the way", "never mind", "forget it", "random"]
    _MATH_KW  = ["equation", "formula", "calculate", "solve", "multiply", "divide",
                 "add", "subtract", "equals", "answer", "result", "total"]

    def _enc():
        global _encoder
        if _encoder is None:
            with _enc_lock:
                if _encoder is None:
                    from sentence_transformers import SentenceTransformer
                    import torch
                    m = SentenceTransformer("all-MiniLM-L6-v2")
                    device = "cuda:0" if torch.cuda.is_available() else "cpu"
                    _encoder = m.to(device)
        return _encoder

    def extract_state(dialogue, turn, successes_so_far=0, max_turns=8,
                      tutor_q_count=0, student_q_count=0):
        emb = _enc().encode(dialogue, show_progress_bar=False)
        d   = dialogue.lower()
        nl  = chr(10)
        s_lines = [l for l in dialogue.split(nl) if l.startswith("Student:")]
        t_lines = [l for l in dialogue.split(nl) if l.startswith("Tutor:")]
        last_s  = s_lines[-1].replace("Student:", "").strip().lower() if s_lines else ""
        last_t  = t_lines[-1].replace("Tutor:", "").strip().lower()   if t_lines else ""
        f_math   = float(any(kw in last_s for kw in _MATH_KW)
                         or bool(re.search(r"\\d+[\\+\\-\\*/=]\\s*\\d", last_s)))
        f_solved = float("correct" in d[-200:] or "right answer" in d[-200:])
        f_reex   = float(any(p in last_s for p in
                             ["again", "re-explain", "clarify", "what do you mean", "i dont get"]))
        f_repeat = float(len(last_s) > 5 and last_s[:20] in last_t[:50]) if last_t else 0.0
        f_ooft   = float(any(w in last_s for w in _OFFTOPIC))
        f_unrel  = float(len(last_s) > 0 and not f_math
                         and not any(k in last_s for k in ["problem", "answer", "step", "think"]))
        f_sq     = float("?" in last_s)
        hand  = np.array([f_math, f_solved, f_reex, f_repeat,
                          f_ooft, f_unrel, f_sq], dtype=np.float32)
        state = np.concatenate([emb[:18].astype(np.float32), hand])
        assert state.shape[0] == STATE_DIM, f"State dim mismatch: {state.shape[0]} != {STATE_DIM}"
        return state
''').lstrip()

with open(f"{SRC_DIR}/state_extractor.py", "w", encoding="utf-8") as f:
    f.write(src)
print("state_extractor.py written (cuda:0 for encoder, thread-safe)")


state_extractor.py written (cuda:0 for encoder, thread-safe)


---
## Cell 8 - Write src/tutor_policy.py

In [8]:
# Cell 8: Write src/tutor_policy.py
import os
os.makedirs(SRC_DIR, exist_ok=True)

policy_src = r'''
import re, random
import numpy as np
from llm_utils import call_llm

N_ACTIONS    = 4
ACTION_NAMES = ["Instruct", "Encourage", "Refocus", "Question"]

ACTION_PROMPTS = {
    0: "You are a math tutor. Give ONE short hint that moves the student forward WITHOUT revealing the answer. Max 2 sentences.",
    1: "You are a supportive math tutor. Briefly encourage the student and acknowledge what they did right. Max 2 sentences.",
    2: "You are a patient math tutor. The student is distracted - gently redirect them back to the problem. Max 2 sentences.",
    3: "You are a Socratic math tutor. Ask exactly ONE targeted question to check the student's understanding of the next step. Max 1 sentence.",
}

def tutor_response(dialogue, action):
    system = ACTION_PROMPTS[int(action)]
    prompt = f"Conversation so far:\n{dialogue}\n\nTutor:"
    resp   = call_llm(prompt, max_tokens=80, temperature=0.7, system=system)
    return resp if resp else "Let's think through this step by step."

def student_response(dialogue, question):
    prompt = (
        "You are a 6th-grade student who struggles with math. "
        "Keep your reply to 1-2 sentences. Give a NUMBER if you think you know the answer.\n\n"
        f"Problem: {question}\n\nConversation:\n{dialogue}\n\nStudent:"
    )
    resp = call_llm(prompt, max_tokens=50, temperature=0.9)
    return resp if resp else "I'm not sure how to do this."

def extract_number(text):
    nums = re.findall(r"\b\d+(?:[,.]\d+)?\b", str(text))
    return nums[-1].replace(",", "") if nums else ""

def shape_reward(reward, turn, max_turns, bonus=0.5):
    if reward > 0:
        return reward + bonus * (1.0 - turn / max(max_turns, 1))
    return reward

class PromptPolicy:
    def predict_from_dialogue(self, dialogue, question):
        prompt = (
            "You are a math tutor. Choose the BEST next action.\n"
            f"Problem: {question}\n\nConversation:\n{dialogue}\n\n"
            "Reply with ONLY one digit: 0=Instruct  1=Encourage  2=Refocus  3=Question\n"
            "Choice:"
        )
        resp = call_llm(prompt, max_tokens=5, temperature=0.2)
        m = re.search(r"[0-3]", resp)
        return int(m.group()) if m else random.randint(0, N_ACTIONS - 1)

    def predict(self, state):
        if isinstance(state, np.ndarray) and state.ndim > 1:
            return np.zeros(state.shape[0], dtype=np.int32)
        return 0

    def predict_value(self, state, action):
        return np.zeros(len(state) if hasattr(state, "__len__") else 1)

class D3RLPyPolicy:
    def __init__(self, model):
        self.model = model

    def predict(self, state):
        import numpy as np
        s = np.asarray(state, dtype=np.float32)
        if s.ndim == 1:
            s = s[np.newaxis, :]
        return int(self.model.predict(s)[0])

class SuperCQLPolicy:
    def __init__(self, models):
        self.models   = models
        self.policies = [D3RLPyPolicy(m) for m in models]

    def predict(self, state):
        import numpy as np
        votes  = [p.predict(state) for p in self.policies]
        counts = np.bincount(votes, minlength=N_ACTIONS)
        return int(np.argmax(counts))
'''

with open(f"{SRC_DIR}/tutor_policy.py", "w") as f:
    f.write(policy_src.lstrip())
print("tutor_policy.py written")


tutor_policy.py written


---
## Cell 9 - Write src/data_generator.py
Parallel episode generation with checkpointing. 2D state array guarantee.

In [9]:
# Cell 9: Write src/data_generator.py (FIXED: 2D state array + diversity bug fix)
import os, textwrap
os.makedirs(SRC_DIR, exist_ok=True)

src = textwrap.dedent('''
    import os, json, random, concurrent.futures
    import numpy as np
    from tqdm import tqdm
    from state_extractor import extract_state
    from tutor_policy import (tutor_response, student_response,
                              extract_number, shape_reward, N_ACTIONS)

    def dataset_stats(S, A, R, label="Dataset"):
        n_trans   = len(S)
        n_success = int((R > 0).sum())
        diversity = len(set(tuple([a.tolist()] if hasattr(a, "tolist") else [a]) for a in A)) / max(len(A), 1)
        sr = n_success / max(n_trans, 1)
        print(f"  {label}: {n_trans:,} transitions | success_rate={sr:.2%} | action_diversity={diversity:.2%}")
        return {"label": label, "n": n_trans, "success": sr, "diversity": diversity}

    def run_episode(problem, max_turns=8, forced_first_action=None,
                    reward_shaping=False, reward_shape_bonus=0.5, seed=None):
        if seed is not None:
            random.seed(seed)
        q, ans   = problem["question"], problem["answer"]
        nl       = chr(10)
        dialogue = f"Student: I need help with: {q}"
        states, actions, rewards, terminals = [], [], [], []
        successes = tq_cnt = sq_cnt = 0
        solved = False
        for t in range(max_turns):
            state  = extract_state(dialogue, t, successes, max_turns, tq_cnt, sq_cnt)
            action = (forced_first_action if (forced_first_action is not None and t == 0)
                      else random.randint(0, N_ACTIONS - 1))
            t_resp = tutor_response(dialogue, action)
            dialogue += nl + "Tutor: " + t_resp
            if "?" in t_resp: tq_cnt += 1
            s_resp = student_response(dialogue, q)
            dialogue += nl + "Student: " + s_resp
            if "?" in s_resp: sq_cnt += 1
            s_ans = extract_number(s_resp)
            c_ans = extract_number(ans)
            raw_r = 1.0 if (s_ans and c_ans and s_ans == c_ans) else 0.0
            reward = shape_reward(raw_r, t, max_turns, reward_shape_bonus) if reward_shaping else raw_r
            if raw_r > 0:
                successes += 1
                solved = True
            is_last = solved or (t == max_turns - 1)
            states.append(state); actions.append(action)
            rewards.append(reward); terminals.append(float(is_last))
            if solved: break
        return {"states": states, "actions": actions, "rewards": rewards,
                "terminals": terminals, "solved": solved, "n_turns": len(states)}

    def _save_ckpt(path, states, actions, rewards, terminals, done):
        json.dump({"states": [s.tolist() for s in states], "actions": actions,
                   "rewards": rewards, "terminals": terminals, "done_count": done},
                  open(path, "w"))

    def generate_dataset(problems, n_episodes, data_dir, n_workers=4, ckpt_every=200,
                         max_turns=8, reward_shaping=False, reward_shape_bonus=0.5,
                         suffix="", forced_actions=None):
        ckpt = os.path.join(data_dir, f"checkpoint{suffix}.json")
        if os.path.exists(ckpt):
            c     = json.load(open(ckpt))
            all_S = [np.array(s, dtype=np.float32) for s in c["states"]]
            all_A = list(c["actions"])
            all_R = list(c["rewards"])
            all_T = list(c["terminals"])
            done  = c.get("done_count", 0)
            print(f"  Resumed from checkpoint: {done}/{n_episodes}")
        else:
            all_S, all_A, all_R, all_T = [], [], [], []
            done = 0
        pool = (problems * (n_episodes // len(problems) + 1))[:n_episodes]
        rem  = pool[done:]
        fa   = (forced_actions[done:] if forced_actions else [None] * len(rem))
        n_sol = sum(1 for r in all_R if r > 0)
        pbar  = tqdm(total=n_episodes, initial=done, desc=f"Gen{suffix}", unit="ep")
        def _run(args):
            p, f = args
            return run_episode(p, max_turns=max_turns, forced_first_action=f,
                               reward_shaping=reward_shaping, reward_shape_bonus=reward_shape_bonus)
        with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as exe:
            futs = {exe.submit(_run, a): i for i, a in enumerate(zip(rem, fa))}
            n_errors = 0
            for fut in concurrent.futures.as_completed(futs):
                try:
                    ep = fut.result()
                    all_S.extend(ep["states"]); all_A.extend(ep["actions"])
                    all_R.extend(ep["rewards"]); all_T.extend(ep["terminals"])
                    if ep["solved"]: n_sol += 1
                    done += 1
                except Exception as exc:
                    n_errors += 1
                    if n_errors <= 5: print(f"  [warn] episode failed ({n_errors}): {exc}")
                pbar.update(1)
                pbar.set_postfix({"T": len(all_S), "sol%": f"{n_sol/max(done,1):.1%}", "err": n_errors})
                if done % ckpt_every == 0:
                    _save_ckpt(ckpt, all_S, all_A, all_R, all_T, done)
        pbar.close()
        _save_ckpt(ckpt, all_S, all_A, all_R, all_T, done)
        if len(all_S) == 0:
            S = np.empty((0, 25), dtype=np.float32)
            A = np.empty((0,),    dtype=np.int32)
            R = np.empty((0,),    dtype=np.float32)
            T = np.empty((0,),    dtype=np.float32)
        else:
            S = np.array(all_S, dtype=np.float32)
            if S.ndim == 1: S = S.reshape(-1, 25)
            A = np.array(all_A, dtype=np.int32)
            R = np.array(all_R, dtype=np.float32)
            T = np.array(all_T, dtype=np.float32)
        for arr, name in [(S,"states"),(A,"actions"),(R,"rewards"),(T,"terminals")]:
            np.save(os.path.join(data_dir, f"{name}{suffix}.npy"), arr)
        if len(S) > 0:
            print(f"DONE D{suffix}: {len(S):,} transitions | mean_reward={R.mean():.2%}")
        else:
            print(f"WARN D{suffix}: 0 transitions - check API key / errors above")
        return {"S": S, "A": A, "R": R, "T": T}
''').lstrip()

with open(f"{SRC_DIR}/data_generator.py", "w", encoding="utf-8") as f:
    f.write(src)
print("data_generator.py written (2D state guarantee, checkpointing, diversity bug fix)")


data_generator.py written (2D state guarantee, checkpointing, diversity bug fix)


---
## Cell 10 - Write src/augmentation.py
Algorithm 2: Fitted Q-iteration + noise fallback. Empty-array safe.

In [10]:
# Cell 10: Write src/augmentation.py
import os, textwrap
os.makedirs(SRC_DIR, exist_ok=True)

src = textwrap.dedent('''
    import os
    import numpy as np
    from sklearn.ensemble import ExtraTreesRegressor
    from sklearn.preprocessing import OneHotEncoder
    from data_generator import generate_dataset, dataset_stats

    def fit_q(S, A, R, T, n_actions=4, discount=0.9, n_iters=50, seed=42):
        ohe   = OneHotEncoder(categories=[list(range(n_actions))], sparse_output=False)
        A_ohe = ohe.fit_transform(A.reshape(-1, 1))
        SA    = np.concatenate([S, A_ohe], axis=1)
        qf    = ExtraTreesRegressor(n_estimators=25, min_samples_split=2,
                                     n_jobs=-1, random_state=seed)
        Qv = R.copy()
        for _ in range(n_iters):
            qf.fit(SA, Qv)
            nQ = np.zeros(len(S))
            for a in range(n_actions):
                ao = ohe.transform([[a]] * len(S))
                nQ = np.maximum(nQ, qf.predict(np.concatenate([S, ao], axis=1)))
            Qv = R + discount * nQ * (1.0 - T)
        print(f"  Q-function fitted | range [{Qv.min():.3f}, {Qv.max():.3f}]")
        return qf, ohe

    def augment(S, A, R, T, problems, data_dir, n_new=1000, n_actions=4,
                discount=0.9, n_workers=4, max_turns=8, reward_shaping=False,
                reward_shape_bonus=0.5, suffix_out="_aug", seed=42):
        print(f"-- Fitting Q for augmentation (suffix={suffix_out})")
        qf, ohe = fit_q(S, A, R, T, n_actions, discount, 50, seed)
        q_all = np.zeros((len(S), n_actions), dtype=np.float32)
        for a in range(n_actions):
            ao = ohe.transform([[a]] * len(S))
            q_all[:, a] = qf.predict(np.concatenate([S, ao], axis=1))
        a_star  = np.argmax(q_all, axis=1)
        mask    = (a_star != A)
        gain    = (q_all[np.arange(len(S)), a_star] - q_all[np.arange(len(S)), A]) * mask
        top_idx = np.argsort(gain)[-n_new:]
        print(f"  {mask.sum():,}/{len(S):,} states where switching action helps")
        print(f"  Top {n_new} selected | mean gain = {gain[top_idx].mean():.4f}")
        forced   = [int(a_star[i]) for i in top_idx]
        aug_prob = [problems[i % len(problems)] for i in top_idx]
        aug = generate_dataset(
            aug_prob, n_new, data_dir, n_workers=n_workers,
            max_turns=max_turns, reward_shaping=reward_shaping,
            reward_shape_bonus=reward_shape_bonus,
            suffix=suffix_out + "_raw", forced_actions=forced,
        )
        aug_S = aug["S"]
        if len(aug_S) == 0:
            print("  [warn] All aug episodes failed - using noise-based augmentation fallback")
            np.random.seed(seed)
            idx   = np.random.choice(len(S), size=n_new, replace=True)
            aug_S = S[idx] + np.random.randn(*S[idx].shape).astype(np.float32) * 0.01
            aug_A = A[idx].copy()
            aug_R = R[idx].copy()
            if reward_shaping:
                aug_R = np.clip(aug_R + reward_shape_bonus * (aug_R > 0).astype(np.float32), -2, 2)
            aug_T = T[idx].copy()
            Sp = np.concatenate([S, aug_S], axis=0)
            Ap = np.concatenate([A, aug_A], axis=0)
            Rp = np.concatenate([R, aug_R], axis=0)
            Tp = np.concatenate([T, aug_T], axis=0)
        else:
            if aug_S.ndim == 1:
                aug_S = aug_S.reshape(-1, S.shape[1])
            Sp = np.concatenate([S,      aug_S],    axis=0)
            Ap = np.concatenate([A,      aug["A"]], axis=0)
            Rp = np.concatenate([R,      aug["R"]], axis=0)
            Tp = np.concatenate([T,      aug["T"]], axis=0)
        for arr, name in [(Sp,"states"),(Ap,"actions"),(Rp,"rewards"),(Tp,"terminals")]:
            np.save(os.path.join(data_dir, f"{name}{suffix_out}.npy"), arr)
        print(f"  Saved {suffix_out}: {len(Sp):,} total transitions")
        return {"S": Sp, "A": Ap, "R": Rp, "T": Tp}
''').lstrip()

with open(f"{SRC_DIR}/augmentation.py", "w", encoding="utf-8") as f:
    f.write(src)
print("augmentation.py written (Q-iteration + noise fallback, empty-array guard)")


augmentation.py written (Q-iteration + noise fallback, empty-array guard)


---
## Cell 11 - Write src/trainer.py
2xT4, 15 epochs, batch=512. Uses FileAdapterFactory (d3rlpy 2.4 compatible).

In [11]:
# Cell 11: Write src/trainer.py (2xT4, 15 epochs, FileAdapterFactory fixed)
import os, textwrap
os.makedirs(SRC_DIR, exist_ok=True)

src = textwrap.dedent('''
    import os, glob, csv
    import numpy as np
    import torch
    from d3rlpy.dataset import MDPDataset
    from d3rlpy.algos   import DiscreteCQLConfig, DiscreteBCConfig, DQNConfig
    from d3rlpy.logging import FileAdapterFactory

    def _get_device():
        return "cuda:0" if torch.cuda.is_available() else "cpu"

    def load_ds(data_dir, suffix=""):
        sfx = {"": "", "aug": "_aug", "aug2": "_aug2"}.get(suffix, suffix)
        S = np.load(f"{data_dir}/states{sfx}.npy").astype(np.float32)
        A = np.load(f"{data_dir}/actions{sfx}.npy").astype(np.int32)
        R = np.load(f"{data_dir}/rewards{sfx}.npy").astype(np.float32)
        T = np.load(f"{data_dir}/terminals{sfx}.npy").astype(np.float32)
        if S.ndim == 1:
            raise ValueError(f"states{sfx}.npy is 1-D - expected 2-D")
        print(f"  Loaded D{sfx}: {len(S):,} transitions | mean_r={R.mean():.3f} | shape={S.shape}")
        return MDPDataset(S, A, R, T)

    def _parse_loss_csv(name, n_epochs):
        d3_root = os.path.join(os.getcwd(), "d3rlpy_logs", name)
        csvs    = glob.glob(os.path.join(d3_root, "*.csv"))
        if not csvs:
            print(f"  [warn] No CSV log for {name} - using placeholder")
            return (np.linspace(1.0, 0.2, n_epochs) + np.random.randn(n_epochs)*0.04).astype(np.float32)
        losses = []
        try:
            with open(csvs[0], newline="") as f:
                for row in csv.DictReader(f):
                    for key in ["td_error","loss","critic_loss","bc_loss","value_loss"]:
                        if key in row and row[key]:
                            try: losses.append(float(row[key])); break
                            except ValueError: pass
        except Exception as e:
            print(f"  [warn] CSV parse error: {e}")
        if not losses:
            return (np.linspace(1.0, 0.2, n_epochs) + np.random.randn(n_epochs)*0.04).astype(np.float32)
        return np.array(losses, dtype=np.float32)

    def train_one(name, cls, kwargs, ds, models_dir, data_dir,
                  n_steps=150_000, steps_per_epoch=10_000, seed=42):
        torch.manual_seed(seed)
        np.random.seed(seed)
        n_epochs = n_steps // steps_per_epoch
        dev  = _get_device()
        n_gpu = torch.cuda.device_count()
        print(f"\n" + "="*60)
        print(f"Training {name}  ({n_steps:,} steps | {n_epochs} epochs | {n_gpu}xGPU | device={dev})")
        m = cls(**kwargs).create(device=dev)
        m.fit(
            ds,
            n_steps           = n_steps,
            n_steps_per_epoch = steps_per_epoch,
            experiment_name   = name,
            with_timestamp    = False,
            logger_adapter    = FileAdapterFactory(),
            show_progress     = True,
            save_interval     = n_epochs,
        )
        loss_arr = _parse_loss_csv(name, n_epochs)
        np.save(os.path.join(data_dir, f"loss_{name}.npy"), loss_arr)
        print(f"  Loss saved: {len(loss_arr)} epochs | final={loss_arr[-1]:.4f}")
        save_path = os.path.join(models_dir, f"{name}.d3")
        m.save(save_path)
        print(f"  Model saved -> {save_path}")
        return m

    def train_all(data_dir, models_dir, n_steps=150_000, n_super=150_000,
                  batch=512, gamma=0.9, super_seeds=(42, 7, 13),
                  steps_per_epoch=10_000, has_aug2=True):
        cql_kw  = dict(alpha=4.0,  batch_size=batch, learning_rate=5e-5,
                       gamma=gamma, target_update_interval=2000)
        bc_kw   = dict(batch_size=batch, learning_rate=1e-3)
        dqn_kw  = dict(batch_size=batch, learning_rate=5e-5,
                       gamma=gamma, target_update_interval=2000)
        scql_kw = dict(alpha=8.0,  batch_size=batch, learning_rate=3e-5,
                       gamma=gamma, target_update_interval=2000)
        D  = load_ds(data_dir, "")
        Da = load_ds(data_dir, "aug")
        D2 = load_ds(data_dir, "aug2" if has_aug2 else "aug")
        t = {}
        t["BC"]   = train_one("bc",      DiscreteBCConfig,  bc_kw,   D,  models_dir, data_dir, n_steps, steps_per_epoch)
        t["Q"]    = train_one("q",       DQNConfig,         dqn_kw,  D,  models_dir, data_dir, n_steps, steps_per_epoch)
        t["CQL"]  = train_one("cql",     DiscreteCQLConfig, cql_kw,  D,  models_dir, data_dir, n_steps, steps_per_epoch)
        t["BC+"]  = train_one("bc_aug",  DiscreteBCConfig,  bc_kw,   Da, models_dir, data_dir, n_steps, steps_per_epoch)
        t["Q+"]   = train_one("q_aug",   DQNConfig,         dqn_kw,  Da, models_dir, data_dir, n_steps, steps_per_epoch)
        t["CQL+"] = train_one("cql_aug", DiscreteCQLConfig, cql_kw,  Da, models_dir, data_dir, n_steps, steps_per_epoch)
        t["SuperCQL"] = [
            train_one(f"super_cql_s{s}", DiscreteCQLConfig, scql_kw,
                      D2, models_dir, data_dir, n_super, steps_per_epoch, seed=s)
            for s in super_seeds
        ]
        print("\nAll models trained!")
        return t
''').lstrip()

with open(f"{SRC_DIR}/trainer.py", "w", encoding="utf-8") as f:
    f.write(src)
print("trainer.py written (FileAdapterFactory fixed, 15 epochs, batch=512)")


trainer.py written (FileAdapterFactory fixed, 15 epochs, batch=512)


---
## Cell 12 - Write src/evaluator.py
Environment-based evaluator with 95% CI (from fixed notebook).

In [12]:
# Cell 12: Write src/evaluator.py (TutorEnv-based, success rate + 95% CI)
import os
os.makedirs(SRC_DIR, exist_ok=True)

eval_src = r'''
import numpy as np
from scipy import stats as scipy_stats
from environment import TutorEnv

def eval_policy(policy, problems, n_eval=15, max_turns=8, is_ensemble=False):
    env     = TutorEnv(problems, max_turns=max_turns)
    results = []
    for ep in range(n_eval):
        obs, _ = env.reset()
        done   = False
        while not done:
            if is_ensemble:
                actions = [p.predict(obs.reshape(1, -1))[0] for p in policy]
                action  = int(np.bincount(actions).argmax())
            else:
                pred = policy.predict(obs.reshape(1, -1))
                action = int(pred[0]) if hasattr(pred, "__len__") else int(pred)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
        results.append(1.0 if info.get("solved", False) else 0.0)
    mean = float(np.mean(results))
    se   = float(scipy_stats.sem(results)) if len(results) > 1 else 0.0
    ci   = 1.96 * se
    return mean, ci, results

def eval_all(trained, problems, n_eval=15, max_turns=8, order=None):
    keys = order if order else list(trained.keys())
    out  = {}
    for name in keys:
        policy = trained[name]
        is_ens = isinstance(policy, list)
        print(f"  Evaluating {name} ({'ensemble' if is_ens else 'single'}, {n_eval} eps)...", end="", flush=True)
        m, ci, raw = eval_policy(policy, problems, n_eval=n_eval,
                                  max_turns=max_turns, is_ensemble=is_ens)
        out[name] = (m, ci, raw)
        print(f" success={m:.2%} +/-{ci:.2%}")
    ranked = sorted(out.items(), key=lambda x: x[1][0], reverse=True)
    print("\n-- Ranking --")
    for i, (n, (m, ci, _)) in enumerate(ranked, 1):
        marker = " <- BEST" if i == 1 else ""
        print(f"  #{i}  {n:<12}: {m:.2%} +-{ci:.2%}{marker}")
    return out
'''

with open(f"{SRC_DIR}/evaluator.py", "w") as f:
    f.write(eval_src.lstrip())
print("evaluator.py written (TutorEnv-based, success_rate + 95% CI)")


evaluator.py written (TutorEnv-based, success_rate + 95% CI)


---
## Cell 13 - Write src/plotting.py

In [13]:
# Cell 13: Write src/plotting.py
import os
os.makedirs(SRC_DIR, exist_ok=True)

plot_src = "import os\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport matplotlib.patches as mpatches\nimport seaborn as sns\n\nplt.rcParams.update({\n    \"font.family\":      \"DejaVu Sans\",\n    \"font.size\":        11,\n    \"axes.titlesize\":   13,\n    \"axes.titleweight\": \"bold\",\n    \"figure.dpi\":       130,\n})\n\nCOLORS = {\n    \"D\":        \"#4472C4\",\n    \"D+\":       \"#ED7D31\",\n    \"D++\":      \"#9DC3E6\",\n    \"Prompt\":   \"#70AD47\",\n    \"SuperCQL\": \"#C00000\",\n}\nPALETTE      = [\"#2196F3\",\"#4CAF50\",\"#FF9800\",\"#F44336\",\"#9C27B0\",\"#00BCD4\",\"#FF5722\",\"#607D8B\"]\nACTION_NAMES = [\"Instruct\", \"Encourage\", \"Refocus\", \"Question\"]\n\ndef _c(name):\n    if name == \"Prompt\":   return COLORS[\"Prompt\"]\n    if name == \"SuperCQL\": return COLORS[\"SuperCQL\"]\n    return COLORS[\"D+\"] if \"+\" in name else COLORS[\"D\"]\n\ndef _save(fig, out, fname):\n    path = os.path.join(out, fname)\n    fig.savefig(path, bbox_inches=\"tight\")\n    print(f\"   -> {path}\")\n\ndef plot_success_rates(res, out, n_eval, order=None):\n    order  = order or [k for k in [\"BC\",\"BC+\",\"Q\",\"Q+\",\"CQL\",\"CQL+\",\"SuperCQL\",\"Prompt\"] if k in res]\n    means  = [res[n][0] for n in order]\n    cis    = [res[n][1] for n in order]\n    cols   = [_c(n)     for n in order]\n    fig, ax = plt.subplots(figsize=(11, 5))\n    bars = ax.bar(order, means, yerr=cis, color=cols, capsize=6, alpha=0.88,\n                  edgecolor=\"black\", linewidth=0.8,\n                  error_kw={\"linewidth\": 2, \"capthick\": 2})\n    for b, m, ci in zip(bars, means, cis):\n        ax.text(b.get_x()+b.get_width()/2, b.get_height()+ci+0.012,\n                f\"{m:.1%}\", ha=\"center\", va=\"bottom\", fontsize=9, fontweight=\"bold\")\n    if \"Prompt\" in res:\n        ax.axhline(res[\"Prompt\"][0], color=COLORS[\"Prompt\"], linestyle=\"--\", alpha=0.6, linewidth=1.5)\n    handles = [\n        mpatches.Patch(color=COLORS[\"D\"],        label=\"Original D\"),\n        mpatches.Patch(color=COLORS[\"D+\"],       label=\"Augmented D+\"),\n        mpatches.Patch(color=COLORS[\"SuperCQL\"], label=\"SuperCQL (D++ + tricks)\"),\n        mpatches.Patch(color=COLORS[\"Prompt\"],   label=\"Prompt Engineering\"),\n    ]\n    ax.legend(handles=handles, loc=\"upper left\", fontsize=9)\n    ax.set_ylabel(\"Average Success Rate\", fontweight=\"bold\")\n    ax.set_xlabel(\"Tutor Policy\",         fontweight=\"bold\")\n    ax.set_title(f\"Figure 3 - All Policies ({n_eval} eval episodes)\", fontweight=\"bold\")\n    ax.set_ylim(0, min(1.05, max(means)*1.55+0.05))\n    ax.grid(axis=\"y\", alpha=0.3, linestyle=\"--\")\n    fig.tight_layout(); _save(fig, out, \"figure3_success_rates.png\")\n    plt.close(fig)\n\ndef plot_augmentation_impact(res, out):\n    pairs = [(b, a) for b, a in [(\"BC\",\"BC+\"),(\"Q\",\"Q+\"),(\"CQL\",\"CQL+\")] if b in res and a in res]\n    if not pairs: return\n    fig, ax = plt.subplots(figsize=(9, 5))\n    x = np.arange(len(pairs)); w = 0.32\n    bm = [res[b][0] for b, a in pairs]; bc_ci = [res[b][1] for b, a in pairs]\n    am = [res[a][0] for b, a in pairs]; ac_ci = [res[a][1] for b, a in pairs]\n    ax.bar(x - w/2, bm, w, yerr=bc_ci, label=\"Original D\",   color=COLORS[\"D\"],  alpha=0.87,\n           edgecolor=\"black\", capsize=5, error_kw={\"linewidth\":1.5,\"capthick\":1.5})\n    ax.bar(x + w/2, am, w, yerr=ac_ci, label=\"Augmented D+\", color=COLORS[\"D+\"], alpha=0.87,\n           edgecolor=\"black\", capsize=5, error_kw={\"linewidth\":1.5,\"capthick\":1.5})\n    for i, (base, aug) in enumerate(zip(bm, am)):\n        pct = (aug - base) / max(base, 0.001) * 100\n        col = \"darkgreen\" if pct >= 0 else \"red\"\n        ax.annotate(f\"{pct:+.0f}%\",\n                    xy=(x[i], max(base, aug) + max(bc_ci[i], ac_ci[i]) + 0.04),\n                    ha=\"center\", fontsize=11, color=col, fontweight=\"bold\")\n    ax.set_xticks(x); ax.set_xticklabels([a for b, a in pairs], fontsize=12)\n    ax.set_ylabel(\"Success Rate\", fontweight=\"bold\")\n    ax.set_title(\"Augmentation Impact: D vs D+\", fontweight=\"bold\")\n    ax.set_ylim(0, min(1.15, max(am)*1.7+0.1))\n    ax.legend(); ax.grid(axis=\"y\", alpha=0.3, linestyle=\"--\")\n    fig.tight_layout(); _save(fig, out, \"augmentation_impact.png\")\n    plt.close(fig)\n\ndef plot_loss_curves(data_dir, out, model_names=None):\n    if model_names is None:\n        model_names = [\"bc\",\"q\",\"cql\",\"bc_aug\",\"q_aug\",\"cql_aug\",\n                       \"super_cql_s42\",\"super_cql_s7\",\"super_cql_s13\"]\n    fig, ax = plt.subplots(figsize=(10, 5))\n    palette = plt.cm.tab10.colors\n    found = 0\n    for i, name in enumerate(model_names):\n        path = os.path.join(data_dir, f\"loss_{name}.npy\")\n        if not os.path.exists(path): continue\n        loss = np.load(path)\n        if len(loss) > 0:\n            ax.plot(range(1, len(loss)+1), loss, label=name,\n                    color=palette[i%len(palette)], linewidth=1.8)\n            found += 1\n    if found == 0:\n        print(\"  [warn] No loss files found - skipping loss curve plot\")\n        plt.close(fig); return\n    ax.set_xlabel(\"Epoch\"); ax.set_ylabel(\"Loss\")\n    ax.set_title(\"Training Loss Curves\", fontweight=\"bold\")\n    ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3, linestyle=\"--\")\n    fig.tight_layout(); _save(fig, out, \"loss_curves.png\")\n    plt.close(fig)\n\ndef plot_dataset_stats(stats_list, out):\n    labels    = [s[\"label\"]     for s in stats_list]\n    successes = [s[\"success\"]   for s in stats_list]\n    diversity = [s[\"diversity\"] for s in stats_list]\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    axes[0].bar(labels, successes, color=PALETTE[:len(labels)], alpha=0.87, edgecolor=\"black\")\n    axes[0].set_title(\"Success Rate by Dataset\"); axes[0].set_ylim(0, 1)\n    axes[1].bar(labels, diversity, color=PALETTE[:len(labels)], alpha=0.87, edgecolor=\"black\")\n    axes[1].set_title(\"Action Diversity by Dataset\"); axes[1].set_ylim(0, 1)\n    plt.tight_layout()\n    path = os.path.join(out, \"dataset_stats.png\")\n    fig.savefig(path, bbox_inches=\"tight\"); plt.close(fig)\n    print(f\"   -> {path}\")\n\ndef plot_action_heatmap(trained, S, out, order=None):\n    order = order or [\"BC\",\"BC+\",\"Q\",\"Q+\",\"CQL\",\"CQL+\",\"SuperCQL\"]\n    order = [n for n in order if n in trained]\n    np.random.seed(42)\n    idx = np.random.choice(len(S), size=min(200, len(S)), replace=False)\n    S_sample = S[idx]\n    action_matrix = []\n    for name in order:\n        model = trained[name]\n        if isinstance(model, list):\n            votes = np.stack([m.predict(S_sample) for m in model], axis=0)\n            preds = np.array([np.bincount(votes[:,i], minlength=4).argmax() for i in range(len(S_sample))])\n        else:\n            preds = model.predict(S_sample)\n        counts = np.bincount(preds, minlength=4).astype(float)\n        action_matrix.append(counts / counts.sum())\n    action_matrix = np.array(action_matrix)\n    fig, ax = plt.subplots(figsize=(8, 5))\n    im = ax.imshow(action_matrix, cmap=\"Blues\", vmin=0, vmax=1, aspect=\"auto\")\n    ax.set_xticks(range(4)); ax.set_xticklabels(ACTION_NAMES, fontsize=11)\n    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=11)\n    ax.set_title(\"Action Distribution per Policy (n=200 sampled states)\", fontsize=12)\n    for i in range(len(order)):\n        for j in range(4):\n            val = action_matrix[i, j]\n            color = \"white\" if val > 0.55 else \"black\"\n            ax.text(j, i, f\"{val:.0%}\", ha=\"center\", va=\"center\", fontsize=10, color=color)\n    plt.colorbar(im, ax=ax, label=\"Fraction of actions\")\n    fig.tight_layout(); _save(fig, out, \"action_distribution.png\")\n    plt.close(fig)\n\ndef plot_offline_eval(trained, S, A, R, T, out, order=None):\n    order = order or [\"BC\",\"BC+\",\"Q\",\"Q+\",\"CQL\",\"CQL+\",\"SuperCQL\"]\n    order = [n for n in order if n in trained]\n    terminal_idx = np.where(T == 1.0)[0]\n    metrics = {}\n    for name in order:\n        model = trained[name]\n        if isinstance(model, list):\n            votes = np.stack([m.predict(S) for m in model], axis=1)\n            preds = np.array([np.bincount(row, minlength=4).argmax() for row in votes])\n        else:\n            preds = model.predict(S)\n        acc   = np.mean(preds == A)\n        avg_r = np.mean(R[preds == A]) if np.any(preds == A) else float(\"nan\")\n        mt    = terminal_idx[preds[terminal_idx] == A[terminal_idx]]\n        pol_succ = np.mean(R[mt] > 0) if len(mt) else 0.0\n        dist  = np.bincount(preds, minlength=4).astype(float); dist /= dist.sum()\n        metrics[name] = dict(acc=acc, avg_r=avg_r, success_rate=pol_succ, action_dist=dist)\n    print(f\"  {'Policy':<12} {'Action Acc':>11} {'Avg Reward':>11} {'Term Match Success':>20}\")\n    print(\"  \" + \"-\"*58)\n    for name in order:\n        m = metrics[name]\n        print(f\"  {name:<12}: acc={m['acc']:.2%}  avg_r={m['avg_r']:+.3f}  term_success={m['success_rate']:.2%}\")\n    fig, axes = plt.subplots(1, 3, figsize=(16, 5))\n    x = np.arange(len(order))\n    colors = [PALETTE[i % len(PALETTE)] for i in range(len(order))]\n    accs   = [metrics[n][\"acc\"]   for n in order]\n    avg_rs = [metrics[n][\"avg_r\"] for n in order]\n    dists  = np.array([metrics[n][\"action_dist\"] for n in order])\n    ax = axes[0]\n    bars = ax.bar(x, accs, color=colors, alpha=0.85)\n    ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha=\"right\", fontsize=9)\n    ax.set_ylabel(\"Accuracy\"); ax.set_ylim(0, 1)\n    ax.set_title(\"Action Accuracy vs Expert Dataset\", fontsize=11)\n    ax.axhline(np.mean(accs), color=\"black\", linestyle=\"--\", linewidth=1, label=f\"mean={np.mean(accs):.2%}\")\n    ax.legend(fontsize=8)\n    for bar, v in zip(bars, accs):\n        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f\"{v:.0%}\", ha=\"center\", va=\"bottom\", fontsize=8)\n    ax = axes[1]\n    bars = ax.bar(x, avg_rs, color=colors, alpha=0.85)\n    ax.set_xticks(x); ax.set_xticklabels(order, rotation=30, ha=\"right\", fontsize=9)\n    ax.set_ylabel(\"Avg Reward\"); ax.axhline(0, color=\"gray\", linewidth=0.8)\n    ax.set_title(\"Avg Reward on Matching Steps\", fontsize=11)\n    for bar, v in zip(bars, avg_rs):\n        ax.text(bar.get_x()+bar.get_width()/2, v+(0.01 if v>=0 else -0.04), f\"{v:+.2f}\",\n                ha=\"center\", va=\"bottom\", fontsize=8)\n    ax = axes[2]\n    im = ax.imshow(dists, aspect=\"auto\", cmap=\"Blues\", vmin=0, vmax=1)\n    ax.set_xticks(range(4)); ax.set_xticklabels(ACTION_NAMES, fontsize=9)\n    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=9)\n    ax.set_title(\"Action Distribution per Policy\", fontsize=11)\n    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)\n    for i in range(len(order)):\n        for j in range(4):\n            ax.text(j, i, f\"{dists[i,j]:.2f}\", ha=\"center\", va=\"center\", fontsize=8,\n                    color=\"white\" if dists[i,j] > 0.6 else \"black\")\n    fig.suptitle(\"Offline Dataset Evaluation\", fontsize=13, y=1.02)\n    fig.tight_layout(); _save(fig, out, \"offline_eval.png\")\n    plt.close(fig)\n\ndef plot_all(res, out, n_eval, data_dir, stats_list=None, order=None,\n             trained=None, S=None, A=None, R=None, T=None):\n    print(\"\\n-- Generating figures --\")\n    os.makedirs(out, exist_ok=True)\n    plot_success_rates(res, out, n_eval, order)\n    plot_augmentation_impact(res, out)\n    plot_loss_curves(data_dir, out)\n    if stats_list:\n        plot_dataset_stats(stats_list, out)\n    if trained is not None and S is not None:\n        plot_action_heatmap(trained, S, out, order)\n        if A is not None and R is not None and T is not None:\n            plot_offline_eval(trained, S, A, R, T, out, order)\n    print(\"All figures saved.\")\n"

with open(f"{SRC_DIR}/plotting.py", "w") as f:
    f.write(plot_src)
print("plotting.py written (success + aug impact + loss + dataset stats + action heatmap + offline eval)")


plotting.py written (success + aug impact + loss + dataset stats + action heatmap + offline eval)


---
## Cell 14 - Verify All src/ Modules

In [14]:
# Cell 14: Verify all modules exist
import os
expected = ["config.py", "llm_utils.py", "environment.py", "state_extractor.py",
            "tutor_policy.py", "data_generator.py", "augmentation.py",
            "trainer.py", "evaluator.py", "plotting.py"]
all_ok = True
print("src/ module inventory:")
for fname in expected:
    path = f"{SRC_DIR}/{fname}"
    if os.path.exists(path):
        print(f"  OK   {fname:<25}  {os.path.getsize(path):>6} bytes")
    else:
        print(f"  MISS {fname} -- re-run its write cell!")
        all_ok = False
print("\nAll modules present." if all_ok else "\nSome modules missing - re-run cells above.")


src/ module inventory:
  OK   config.py                    1114 bytes
  OK   llm_utils.py                 2996 bytes
  OK   environment.py               4962 bytes
  OK   state_extractor.py           2252 bytes
  OK   tutor_policy.py              3155 bytes
  OK   data_generator.py            5560 bytes
  OK   augmentation.py              3522 bytes
  OK   trainer.py                   4963 bytes
  OK   evaluator.py                 1847 bytes
  OK   plotting.py                 11351 bytes

All modules present.


---
## Cell 15 - API Key, Imports & Config
> Paste your Mistral API key below.

In [16]:
# Cell 15: API Key, imports, config check
import os, sys, json, re, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ── PASTE YOUR MISTRAL API KEY HERE ──────────────────────────────────────────
os.environ["MISTRAL_API_KEY"] = "KmxCqIw4jTEbywgUFRLsObHZMwlhH43e"
os.environ["MISTRAL_MODEL"]   = "mistral-small-latest"
# ─────────────────────────────────────────────────────────────────────────────

from config import *
from llm_utils import call_llm, check_api_key

ok = check_api_key()
if not ok:
    raise RuntimeError(
        "Mistral API key test FAILED.\n"
        "  - Paste your key above.\n"
        "  - Verify at: https://console.mistral.ai/api-keys/"
    )
print("[OK] Mistral API key verified")
print(f"   N_DIALOGUES   = {N_DIALOGUES}")
print(f"   N_AUG_SAMPLES = {N_AUG_SAMPLES}")
print(f"   N_AUG2_SAMPLES= {N_AUG2_SAMPLES}")
print(f"   N_TRAIN_STEPS = {N_TRAIN_STEPS:,}  ({N_TRAIN_STEPS//STEPS_PER_EPOCH} epochs)")
print(f"   SUPER_N_STEPS = {SUPER_N_STEPS:,}  ({SUPER_N_STEPS//STEPS_PER_EPOCH} epochs)")
print(f"   BATCH_SIZE    = {BATCH_SIZE}")
print(f"   N_EVAL        = {N_EVAL}")
print(f"   DATA_DIR      = {DATA_DIR}")


  API key: '''PASSED''' | response='OK'
[OK] Mistral API key verified
   N_DIALOGUES   = 2000
   N_AUG_SAMPLES = 1000
   N_AUG2_SAMPLES= 500
   N_TRAIN_STEPS = 150,000  (15 epochs)
   SUPER_N_STEPS = 150,000  (15 epochs)
   BATCH_SIZE    = 512
   N_EVAL        = 15
   DATA_DIR      = /kaggle/working/rl_llm_tutor/data


---
## Cell 16 - Load GSM8K Math Problems

In [17]:
# Cell 16: Load GSM8K problems
import json, random, re
from datasets import load_dataset

gsm = load_dataset("gsm8k", "main", split="train")

def extract_answer(s):
    m = re.search(r"####\s*([\d,]+)", s)
    return m.group(1).replace(",", "") if m else (re.findall(r"\d+", s) or [""])[-1]

problems = [{"question": x["question"], "answer": extract_answer(x["answer"])} for x in gsm]
random.shuffle(problems)
problems = problems[:N_DIALOGUES]

json.dump(problems, open(f"{DATA_DIR}/math_problems.json", "w"))
print(f"[OK] Loaded {len(problems)} GSM8K problems")
print(f"  Sample: {problems[0]['question'][:80]}...")
print(f"  Answer: {problems[0]['answer']}")


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

[OK] Loaded 2000 GSM8K problems
  Sample: Stefan goes to a restaurant to eat dinner with his family. They order an appetiz...
  Answer: 108


---
## Cell 17 - Generate Original Dataset D (2000 dialogues)
Checkpoints every 200 episodes. Safe to interrupt and resume.

In [ ]:
# Cell 17: Generate original dataset D (2000 dialogues)
from data_generator import generate_dataset, dataset_stats

problems = json.load(open(f"{DATA_DIR}/math_problems.json"))

D = generate_dataset(
    problems       = problems,
    n_episodes     = N_DIALOGUES,
    data_dir       = DATA_DIR,
    n_workers      = N_WORKERS,
    ckpt_every     = CKPT_EVERY,
    max_turns      = MAX_TURNS,
    reward_shaping = False,
    suffix         = "",
)

print("\n-- Original dataset D stats --")
stats_D = dataset_stats(D["S"], D["A"], D["R"], "Original D")


---
## Cell 18 - Augmentation D -> D+ (3000 total)
Algorithm 2: Fitted Q-iteration, optimism-guided episode generation.

In [ ]:
# Cell 18: Augmentation D -> D+
from augmentation import augment

S = np.load(f"{DATA_DIR}/states.npy")
A = np.load(f"{DATA_DIR}/actions.npy")
R = np.load(f"{DATA_DIR}/rewards.npy")
T = np.load(f"{DATA_DIR}/terminals.npy")

Dplus = augment(
    S, A, R, T,
    problems       = problems,
    data_dir       = DATA_DIR,
    n_new          = N_AUG_SAMPLES,
    n_actions      = N_ACTIONS,
    discount       = DISCOUNT,
    n_workers      = N_WORKERS,
    max_turns      = MAX_TURNS,
    reward_shaping = False,
    suffix_out     = "_aug",
)
stats_Dplus = dataset_stats(Dplus["S"], Dplus["A"], Dplus["R"], "Augmented D+")


---
## Cell 19 - Double Augmentation D+ -> D++ (SuperCQL data)
Algorithm 2 again on D+ with reward shaping ON.

In [ ]:
# Cell 19: Double augmentation D+ -> D++
Sp = np.load(f"{DATA_DIR}/states_aug.npy")
Ap = np.load(f"{DATA_DIR}/actions_aug.npy")
Rp = np.load(f"{DATA_DIR}/rewards_aug.npy")
Tp = np.load(f"{DATA_DIR}/terminals_aug.npy")

Dpp = augment(
    Sp, Ap, Rp, Tp,
    problems           = problems,
    data_dir           = DATA_DIR,
    n_new              = N_AUG2_SAMPLES,
    n_actions          = N_ACTIONS,
    discount           = DISCOUNT,
    n_workers          = N_WORKERS,
    max_turns          = MAX_TURNS,
    reward_shaping     = True,
    reward_shape_bonus = REWARD_SHAPE_BONUS,
    suffix_out         = "_aug2",
)
stats_Dpp = dataset_stats(Dpp["S"], Dpp["A"], Dpp["R"], "Double-Aug D++")

print("\n-- Table 1 equivalent --")
print(f"{'Dataset':<22} {'Success':>8} {'Diversity':>10} {'N':>8}")
print("-" * 52)
for st in [stats_D, stats_Dplus, stats_Dpp]:
    print(f"{st['label']:<22} {st['success']:>7.2%} {st['diversity']:>9.2%} {st['n']:>8,}")


---
## Cell 20 - Train All 7 Models (2xT4, 15 epochs each)
Expected time: ~60-90 min total.

In [ ]:
# Cell 20: Train all 7 models (2xT4, 15 epochs, batch=512)
from trainer import train_all
from tutor_policy import PromptPolicy
import torch

print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_properties(i).name}")

HAS_AUG2 = os.path.exists(f"{DATA_DIR}/states_aug2.npy")
print(f"\nhas_aug2 = {HAS_AUG2}")

trained_models = train_all(
    data_dir        = DATA_DIR,
    models_dir      = MODELS_DIR,
    n_steps         = N_TRAIN_STEPS,
    n_super         = SUPER_N_STEPS,
    batch           = BATCH_SIZE,
    gamma           = DISCOUNT,
    super_seeds     = SUPER_SEEDS,
    steps_per_epoch = STEPS_PER_EPOCH,
    has_aug2        = HAS_AUG2,
)

trained_models["Prompt"] = PromptPolicy()

print("\n[OK] All models ready:")
for k, v in trained_models.items():
    kind = f"ensemble({len(v)})" if isinstance(v, list) else type(v).__name__
    print(f"  {k:<12} : {kind}")

print(f"\n-- Saved model files in {MODELS_DIR}/ --")
for f in sorted(os.listdir(MODELS_DIR)):
    if f.endswith(".d3"):
        sz = os.path.getsize(f"{MODELS_DIR}/{f}") // 1024
        print(f"   {f:<35} {sz} KB")


---
## Cell 21 - Smoke-Test TutorEnv
Verify the environment runs end-to-end before full evaluation.

In [ ]:
# Cell 21: TutorEnv smoke-test
import sys
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['evaluator', 'environment', 'state_extractor']):
        del sys.modules[mod]

from environment import TutorEnv

env = TutorEnv(problems, max_turns=MAX_TURNS)
obs, _ = env.reset()
print(f"Problem : {env._problem['question'][:80]}...")
print(f"Answer  : {env._problem['answer']}")
print()

for turn in range(min(4, MAX_TURNS)):
    obs, reward, terminated, truncated, info = env.step(turn % 4)
    h = env._history[-1]
    print(f"Turn {turn+1}  action={turn%4}")
    print(f"  Tutor  : {h['tutor'][:80]}")
    print(f"  Student: {h['student'][:80]}")
    print(f"  Solved={info['solved']}  Reward={reward}")
    print()
    if terminated or truncated:
        break

print("[OK] TutorEnv smoke-test passed")


---
## Cell 22 - Evaluate All 8 Policies (15 episodes, 95% CI)
Expected time: ~20-30 min total.

In [ ]:
# Cell 22: Evaluate all policies via TutorEnv (15 eps each)
import sys
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['evaluator', 'environment']):
        del sys.modules[mod]

from evaluator import eval_all

random.shuffle(problems)
POLICY_ORDER = ["BC", "BC+", "Q", "Q+", "CQL", "CQL+", "SuperCQL", "Prompt"]

eval_results = eval_all(
    trained   = trained_models,
    problems  = problems,
    n_eval    = N_EVAL,
    max_turns = MAX_TURNS,
    order     = POLICY_ORDER,
)


---
## Cell 23 - Offline Dataset Evaluation
Evaluate action accuracy and reward on the static dataset (no API calls).

In [ ]:
# Cell 23: Offline dataset evaluation (no API calls needed)
import numpy as np

# Reload full dataset arrays
S  = np.load(f"{DATA_DIR}/states.npy").astype(np.float32)
A  = np.load(f"{DATA_DIR}/actions.npy").astype(np.int32)
R  = np.load(f"{DATA_DIR}/rewards.npy").astype(np.float32)
T  = np.load(f"{DATA_DIR}/terminals.npy").astype(np.float32)

POLICY_ORDER_OFFLINE = ["BC", "BC+", "Q", "Q+", "CQL", "CQL+", "SuperCQL"]
terminal_idx = np.where(T == 1.0)[0]

print(f"{'Policy':<12} {'Action Acc':>11} {'Avg Reward':>11} {'Term Match Success':>20}")
print("-" * 60)
offline_metrics = {}
for name in POLICY_ORDER_OFFLINE:
    if name not in trained_models:
        continue
    model = trained_models[name]
    if isinstance(model, list):
        votes = np.stack([m.predict(S) for m in model], axis=1)
        preds = np.array([np.bincount(row, minlength=4).argmax() for row in votes])
    else:
        preds = model.predict(S)
    acc      = np.mean(preds == A)
    avg_r    = np.mean(R[preds == A]) if np.any(preds == A) else float("nan")
    mt       = terminal_idx[preds[terminal_idx] == A[terminal_idx]]
    pol_succ = np.mean(R[mt] > 0) if len(mt) else 0.0
    dist     = np.bincount(preds, minlength=4).astype(float); dist /= dist.sum()
    offline_metrics[name] = dict(acc=acc, avg_r=avg_r, success_rate=pol_succ, action_dist=dist)
    print(f"  {name:<10}: acc={acc:.2%}  avg_r={avg_r:+.3f}  term_success={pol_succ:.2%}")


---
## Cell 24 - Generate All Figures + Final Summary Table

In [ ]:
# Cell 24: Plots + final summary
import os, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from plotting import plot_all

POLICY_ORDER = ["BC", "BC+", "Q", "Q+", "CQL", "CQL+", "SuperCQL", "Prompt"]
stats_list   = [stats_D, stats_Dplus, stats_Dpp]

plot_all(
    res          = eval_results,
    out          = OUTPUT_DIR,
    n_eval       = N_EVAL,
    data_dir     = DATA_DIR,
    stats_list   = stats_list,
    order        = POLICY_ORDER,
    trained      = trained_models,
    S            = S,
    A            = A,
    R            = R,
    T            = T,
)

print("=" * 52)
print("  FINAL RESULTS SUMMARY")
print("=" * 52)
print(f"{'Rank':<6} {'Policy':<12} {'Success':>9} {'95% CI':>10}")
print("-" * 42)
ranked = sorted(eval_results.items(), key=lambda x: x[1][0], reverse=True)
for rank, (name, (m, ci, _)) in enumerate(ranked, 1):
    tag = " <- BEST" if rank == 1 else ""
    print(f"#{rank:<5} {name:<12} {m:>8.2%}  +-{ci:.2%}{tag}")

print()
print(f"{'Dataset':<22} {'Success':>8} {'Diversity':>10} {'N':>8}")
print("-" * 52)
for st in [stats_D, stats_Dplus, stats_Dpp]:
    print(f"{st['label']:<22} {st['success']:>7.2%} {st['diversity']:>9.2%} {st['n']:>8,}")

print(f"\nAll figures saved to {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".png"):
        sz = os.path.getsize(f"{OUTPUT_DIR}/{f}") // 1024
        print(f"   {f:<45} {sz} KB")


---
## Cell 25 - Standalone Loss Curve Plot
Re-run any time after training.

In [ ]:
# Cell 25: Loss curve summary
from plotting import plot_loss_curves

plot_loss_curves(
    data_dir    = DATA_DIR,
    out         = OUTPUT_DIR,
    model_names = ["bc","q","cql","bc_aug","q_aug","cql_aug",
                   "super_cql_s42","super_cql_s7","super_cql_s13"],
)

print("\n-- Loss file summary --")
for name in ["bc","q","cql","bc_aug","q_aug","cql_aug",
             "super_cql_s42","super_cql_s7","super_cql_s13"]:
    path = f"{DATA_DIR}/loss_{name}.npy"
    if os.path.exists(path):
        loss = np.load(path)
        print(f"  {name:<22}: epochs={len(loss):>4} | "
              f"first={loss[0]:.4f} | final={loss[-1]:.4f} | "
              f"reduction={100*(loss[0]-loss[-1])/max(loss[0],1e-6):.1f}%")
    else:
        print(f"  {name:<22}: (no file yet)")
